In [ ]:
# Notebook: Complete Documentation - From State-Space to Transfer Function and Back via Similarity Transformation
import sympy as sp

# Enable LaTeX display formatting
sp.init_printing(use_latex='mathjax')

# ==========================================
# PART 1: FROM INITIAL STATE-SPACE TO TRANSFER FUNCTION
# ==========================================
print("--- PART 1: Initial State-Space Description to Transfer Function ---")

# Define initial system matrices from the provided text description
A_old = sp.Matrix([[-1, 0, 0], 
                   [ 0, -3, 0], 
                   [ 0,  0, -2]])
B_old = sp.Matrix([1, 1, 1])
C_old = sp.Matrix([[1, 2, 1]])
D_old = sp.Matrix([[0]])

print("Initial Matrix A:")
display(A_old)
print("Initial Matrix B:")
display(B_old)
print("Initial Matrix C:")
display(C_old)
print("Initial Matrix D:")
display(D_old)

# Compute the transfer function using the relation H(s) = C * (s*I - A)^-1 * B + D
s = sp.symbols('s')
I3 = sp.eye(3)
H_expr_initial = C_old * (s * I3 - A_old).inv() * B_old + D_old
H_s_derived = sp.cancel(H_expr_initial[0, 0])

print("\nStarting from these initial matrices, we derived the following transfer function:")
display(sp.Eq(sp.Symbol('\\mathcal{H}(s)'), H_s_derived))


# ==========================================
# PART 2: FROM TRANSFER FUNCTION TO CONTROLLER/OBSERVER FORMS
# ==========================================
print("\n--- PART 2: From Transfer Function back to State-Space Canonical Forms ---")

# Now, taking this transfer function, we construct the state-space realizations
# H(s) from the image/problem: (3s^3 + 22s^2 + 48s + 31) / (s^3 + 6s^2 + 11s + 6)
num_given = 3*s**3 + 22*s**2 + 48*s + 31
den_given = s**3 + 6*s**2 + 11*s + 6
H_s = num_given / den_given

print("Target Transfer Function:")
display(H_s)

# Extract Feedthrough Term D and Strictly Proper Part
D_val = sp.limit(H_s, s, sp.oo)
H_strict = sp.cancel(H_s - D_val)
num_strict, den_strict = sp.fraction(H_strict)

# Extract Denominator Coefficients (alpha coefficients)
alpha_2 = den_strict.coeff(s, 2)
alpha_1 = den_strict.coeff(s, 1)
alpha_0 = den_strict.subs(s, 0)

# Extract Numerator Coefficients for Strict Proper Part (beta coefficients)
beta_2 = num_strict.coeff(s, 2)
beta_1 = num_strict.coeff(s, 1)
beta_0 = num_strict.subs(s, 0)

# Controllability Form (Controller Normal Form)
A_c = sp.Matrix([[0, 1, 0],
                 [0, 0, 1],
                 [-alpha_0, -alpha_1, -alpha_2]])
B_c = sp.Matrix([0, 0, 1])
C_c = sp.Matrix([[beta_0, beta_1, beta_2]])
D_c = sp.Matrix([[D_val]])

print("\nDerived Controllability Form Matrices from the Transfer Function:")
print("Matrix A_c:")
display(A_c)
print("Matrix B_c:")
display(B_c)
print("Matrix C_c:")
display(C_c)
print("Matrix D_c:")
display(D_c)


# ==========================================
# PART 3: SIMILARITY TRANSFORMATION VERIFICATION
# ==========================================
print("\n--- PART 3: Similarity Transformation Verification ---")

# Compute controllability matrices for old and new systems
M_c_old = sp.Matrix.hstack(B_old, A_old * B_old, A_old**2 * B_old)
M_c_new = sp.Matrix.hstack(B_c, A_c * B_c, A_c**2 * B_c)

# Find the similarity transformation matrix T such that A_c = T^(-1) * A_old * T
T = M_c_old * M_c_new.inv()

print("Transformation Matrix T:")
display(T)

# Verify similarity transformation relations
A_check = sp.simplify(T.inv() * A_old * T)
B_check = sp.simplify(T.inv() * B_old)
C_check = sp.simplify(C_old * T)

# Documentation output as comments and print statements
print("\n--- Documentation & Conclusions ---")
print("COMMENT: Starting from the initial system matrices, we obtained the transfer function.")
print("COMMENT: Conversely, starting from that transfer function, we constructed the new canonical state-space pair.")
print("COMMENT: Although these pairs of matrices are not identical, they are not arbitrary:")
print("COMMENT: they are strictly connected via a similarity transformation T.")
print("COMMENT: Thus, they describe the exact same dynamic system and produce the same transfer function.")
print("Does A_c == T^(-1) * A_old * T?", A_check == A_c)
print("Does B_c == T^(-1) * B_old?", B_check == B_c)
print("Does C_c == C_old * T?", C_check == C_c)